In [ ]:
import pandas as pd
import numpy as np
import random

# --- CONFIGURACIÓN ---
NUM_REGISTROS = 80000
SEMILLA = 99
np.random.seed(SEMILLA)
random.seed(SEMILLA)

# --- BASE DE CONOCIMIENTO AGRÍCOLA (Rendimientos por Tarea) ---
# Formato: 'Producto': (Promedio_Rendimiento, 'Unidad_Medida')
yield_db = {
    # Granos
    "Arroz": (5.0, "Quintales (QQ)"),
    "Maíz": (4.0, "Quintales (QQ)"),
    "Habichuelas Rojas": (2.0, "Quintales (QQ)"),
    "Habichuelas Negras": (2.0, "Quintales (QQ)"),
    "Guandules": (1.5, "Quintales (QQ)"),
    "Maní": (2.5, "Quintales (QQ)"),

    # Musáceas (Unidades por tarea aprox)
    "Plátano": (350, "Unidades"), 
    "Guineo": (500, "Unidades"),
    "Rulo": (400, "Unidades"),

    # Raíces y Tubérculos
    "Yuca": (20, "Quintales (QQ)"),
    "Batata": (15, "Quintales (QQ)"),
    "Ñame": (12, "Quintales (QQ)"),
    "Yautía Coco": (10, "Quintales (QQ)"),
    "Yautía Blanca": (10, "Quintales (QQ)"),
    "Papa": (25, "Quintales (QQ)"),
    "Mapuey": (10, "Quintales (QQ)"),

    # Vegetales (Libras o Unidades según mercado estándar)
    "Tomate de Ensalada": (4000, "Libras"),
    "Tomate Industrial": (5000, "Libras"),
    "Ají Cubanela": (2500, "Libras"),
    "Ají Morrón": (2200, "Libras"),
    "Vainita": (1500, "Libras"),
    "Berenjena": (1500, "Unidades"), # A veces por docena, usaremos unidades
    "Molondrón": (1200, "Libras"),
    "Pepino": (3000, "Unidades"),
    "Auyama": (1500, "Libras"),
    "Tayota": (2000, "Unidades"),

    # Hortalizas de Hoja/Bulbo
    "Cebolla Roja": (20, "Quintales (QQ)"),
    "Ajo": (12, "Quintales (QQ)"),
    "Repollo": (1000, "Unidades"),
    "Lechuga": (1200, "Unidades"),
    "Zanahoria": (20, "Quintales (QQ)"),
    "Remolacha": (20, "Quintales (QQ)"),
    "Brócoli": (15, "Quintales (QQ)"),
    "Coliflor": (15, "Quintales (QQ)"),
    "Cilantro": (800, "Paquetes"),

    # Frutales
    "Aguacate": (300, "Unidades"),
    "Mango": (500, "Unidades"), # Arboles adultos
    "Chinola": (3000, "Unidades"),
    "Lechosa": (2500, "Libras"),
    "Piña": (2000, "Unidades"), # Alta densidad
    "Limón Persa": (1000, "Unidades"),
    "Limón Agrio": (1200, "Unidades"),
    "Naranja Dulce": (800, "Unidades"),
    "Naranja Agria": (800, "Unidades"),
    "Zapote": (200, "Unidades"),
    "Cereza": (2000, "Libras"),
    "Melón": (1500, "Unidades"),
    "Sandía": (800, "Unidades"),
    "Coco": (150, "Unidades"), # Por tarea (aprox 15 matas x 10 cocos/mes)

    # Agroindustriales
    "Cacao": (1.2, "Quintales (QQ)"),
    "Café": (1.0, "Quintales (QQ)"),
    "Tabaco": (2.0, "Quintales (QQ)"),
    "Caña de Azúcar": (5, "Toneladas") # Excepción: Caña se mide en Toneladas
}

# Listas auxiliares
lista_productos = list(yield_db.keys())
regiones_clima = {
    "Norte": ["Tropical Húmedo", "Templado"],
    "Sur": ["Semiárido", "Bosque Seco"],
    "Este": ["Tropical de Sabana"]
}
preparaciones = ["Arado Mecanizado", "Arado Bueyes", "Corte y Quema", "Siembra Directa", "Invernadero"]
anios = [2020, 2021, 2022, 2023, 2024]

# --- GENERACIÓN ---
data = []

for _ in range(NUM_REGISTROS):

    # 1. Selección de Dimensiones
    prod_nombre = random.choice(lista_productos)
    region = random.choice(list(regiones_clima.keys()))
    clima = random.choice(regiones_clima[region])
    prep = random.choice(preparaciones)
    anio = random.choice(anios)
    mes = random.randint(1, 12)

    # 2. Definir Área (Log-normal para simular muchos pequeños y pocos grandes)
    # Rango aprox: 5 a 500 tareas
    area_tareas = int(np.random.lognormal(3.0, 0.8) * 3) 
    if area_tareas < 2: area_tareas = 2 # Mínimo 2 tareas

    # 3. Lógica de Producción con Margen de Error Controlado
    # Obtenemos datos base del diccionario
    rendimiento_base, unidad = yield_db[prod_nombre]

    # FACTOR DE VARIABILIDAD (El "Margen de error de 80%")
    # Generamos un número entre 0.20 (muy malo) y 1.80 (muy bueno)
    # Centrado en 1.0 (promedio)
    factor_variabilidad = np.random.uniform(0.2, 1.8)

    # Ajuste por tecnología (Invernadero sube rendimiento, Bueyes baja un poco)
    factor_tecnologia = 1.0
    if prep == "Invernadero" and unidad in ["Libras", "Unidades"]:
        factor_tecnologia = 1.5 # Invernadero rinde 50% más en vegetales
    elif prep == "Corte y Quema":
        factor_tecnologia = 0.8 # Método rudimentario rinde menos

    # CALCULO FINAL DE PRODUCCIÓN
    produccion = area_tareas * rendimiento_base * factor_variabilidad * factor_tecnologia

    # Redondeo limpio
    if produccion < 1: produccion = 1
    produccion = round(produccion, 2)

    # 4. Precio (Inverso a la producción + aleatoriedad)
    # Precio base dummy, ajustado
    precio_base = random.uniform(20, 3000) # Genérico
    # Ajuste realista simplificado según unidad
    if unidad == "Quintales (QQ)": precio_base = random.uniform(1500, 4000)
    elif unidad == "Libras": precio_base = random.uniform(10, 60)
    elif unidad == "Unidades": precio_base = random.uniform(5, 50)
    elif unidad == "Toneladas": precio_base = random.uniform(1800, 2500)

    # Si hay mucha producción (temporada alta), el precio baja ligeramente
    factor_oferta = 1.0
    if factor_variabilidad > 1.2: factor_oferta = 0.9

    precio_final = round(precio_base * factor_oferta, 2)

    # Guardar registro
    data.append([
        anio, mes, region, clima, prod_nombre, prep,
        area_tareas, unidad, produccion, precio_final
    ])

# Crear DataFrame
df_agro = pd.DataFrame(data, columns=[
    "Año", "Mes", "Región", "Clima", "Producto", 
    "Preparación_Suelo", "Área_Sembrada_Tareas", 
    "Unidad_Medida", "Producción_Total", "Precio_Unitario"
])

# Verificación de lógica (Mostrar un ejemplo)
print("Ejemplo de consistencia de datos:")
ejemplo_arroz = df_agro[df_agro['Producto'] == 'Arroz'].iloc[0]
rendimiento_real = ejemplo_arroz['Producción_Total'] / ejemplo_arroz['Área_Sembrada_Tareas']
print(f"Producto: Arroz | Área: {ejemplo_arroz['Área_Sembrada_Tareas']} tareas | Producción: {ejemplo_arroz['Producción_Total']} QQ")
print(f"Rendimiento calculado: {rendimiento_real:.2f} QQ/Tarea (Debe estar entre 1.0 y 9.0 aprox)")

# Exportar
df_agro.to_csv("dataset_agro_rd_validado.csv", index=False)
print(f"\nDataset generado con {NUM_REGISTROS} registros.")

